# Loading the Required Libraries

In [1]:
!pip install -qU bitsandbytes transformers datasets accelerate loralib einops xformers
!pip install -q -U git+https://github.com/huggingface/peft.git
!pip install rouge-score nltk datasets
!pip install evaluate

import os
import bitsandbytes as bnb
from google.colab import drive
import pandas as pd
import torch
import torch.nn as nn
import transformers
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset
from peft import (
    LoraConfig,
    PeftConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import evaluate
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import numpy as np
import nltk
nltk.download('punkt')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Loading the Pre-Trained Model

In [2]:
model_id = "PrunaAI/ruslanmv-Medical-Llama3-8B-bnb-4bit-smashed"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    load_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained("ruslanmv/Medical-Llama3-8B")
tokenizer.pad_token = tokenizer.eos_token

Unused kwargs: ['load_4bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.
/usr/local/lib/python3.10/dist-packages/transformers/quantizers/auto.py:186: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# Preparing the Model for QLoRA

In [3]:
model = prepare_model_for_kbit_training(model)

# Configuring LoRA

In [4]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

# Loading and Preparing the Dataset

In [5]:
# Load the CSV file
df = pd.read_csv('/kaggle/input/covid-dataset-part-1-frensh/covid_dataset_part_1_frensh.csv')

# df = pd.read_csv(r"D:\My Files\Career\Engineering\3\S1\NLP\Projet\Traduction Test\Data Split\Output\merged_covid_dataset_french.csv")

# Split the dataset into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Convert DataFrames to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def generate_prompt(data_point):
    return f"""
<Human>: {data_point["Question"]}
<AI>: {data_point["Answer"]}
    """.strip()

def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(data_point)
    tokenized_full_prompt = tokenizer(full_prompt, padding=True, truncation=True)
    return tokenized_full_prompt

# Process the datasets
train_dataset = train_dataset.shuffle().map(generate_and_tokenize_prompt)
test_dataset = test_dataset.shuffle().map(generate_and_tokenize_prompt)

Map:   0%|          | 0/1788 [00:00<?, ? examples/s]

Map:   0%|          | 0/448 [00:00<?, ? examples/s]

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2236 entries, 0 to 2235
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Answer    2236 non-null   object
 1   Question  2236 non-null   object
dtypes: object(2)
memory usage: 35.1+ KB


# Setting Up the Training Arguments

In [7]:
training_args = transformers.TrainingArguments(
    # auto_find_batch_size=True,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    # num_train_epochs=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    # bf16=True,
    save_total_limit=4,
    logging_steps=10,
    output_dir='medical-llama-covid-french-finetuned',
    save_strategy='epoch',
    evaluation_strategy="steps",  # Add evaluation every step
    eval_steps=100,              # Evaluate every 100 steps
    warmup_ratio=0.1,           # Add warmup
    gradient_accumulation_steps=8,
    fp16=True,                  # Enable mixed precision training
    report_to="tensorboard",  # Change this to enable logging
    logging_dir="./logs",     # Add this to specify logging directory
    logging_first_step=True,  # Add this to log the first step
    logging_strategy="steps", # Add this to ensure regular logging
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# Training the Model

In [8]:
trainer = transformers.Trainer(
    model=model,
    train_dataset=train_dataset,      # Use the train split
    eval_dataset=test_dataset,        # Add the test split for evaluation
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Disable caching for training
model.config.use_cache = False

# Start training
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,1.601600,1.549349
200,1.455800,1.511455


TrainOutput(global_step=223, training_loss=1.6105694631824579, metrics={'train_runtime': 4465.5103, 'train_samples_per_second': 0.4, 'train_steps_per_second': 0.05, 'total_flos': 1.41236829321216e+16, 'train_loss': 1.6105694631824579, 'epoch': 0.9977628635346756})

# Evaluate the model

In [9]:
def compute_metrics(model, test_dataset, tokenizer):
    # Initialize metrics
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    # Lists to store predictions and references
    predictions = []
    references = []
    skipped_samples = 0
    
    # Set model to evaluation mode
    model.eval()
    
    # Process each sample individually
    for item in test_dataset:
        try:
            # Extract question from the formatted prompt
            if isinstance(item['Question'], str):
                input_text = f"<Human>: {item['Question'].strip()}"
            else:
                print(f"Skipping invalid question format: {item['Question']}")
                skipped_samples += 1
                continue

            # Prepare input
            inputs = tokenizer(input_text, 
                             return_tensors='pt',
                             padding=True,
                             truncation=True,
                             max_length=512)
            
            input_ids = inputs['input_ids'].to(model.device)
            attention_mask = inputs['attention_mask'].to(model.device)

            # Generate prediction
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_length=512,
                    num_return_sequences=1,
                    no_repeat_ngram_size=2,
                    do_sample=False,
                    temperature=1.0,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            # Decode prediction
            pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            # Clean up prediction by removing the input question
            if "<AI>:" in pred_text:
                pred_text = pred_text.split("<AI>:")[1].strip()
            
            # Get reference answer
            ref_text = item.get('Answer', '').strip()
            if not ref_text:
                print(f"Skipping empty or invalid reference answer for question: {input_text}")
                skipped_samples += 1
                continue

            # Tokenize for BLEU
            try:
                pred_tokens = nltk.word_tokenize(pred_text.lower())
                ref_tokens = nltk.word_tokenize(ref_text.lower())
                
                predictions.append(pred_tokens)
                references.append([ref_tokens])
            except Exception as e:
                print(f"Error tokenizing text: {str(e)}")
                skipped_samples += 1
                continue

        except Exception as e:
            print(f"Error processing sample: {str(e)}")
            skipped_samples += 1
            continue

    if not predictions:
        raise ValueError("No valid predictions generated. Check your dataset and model outputs.")

    print(f"\nProcessed {len(predictions)} samples successfully")
    print(f"Skipped {skipped_samples} samples due to errors")

    try:
        # Calculate BLEU score
        smoothing = SmoothingFunction().method1
        bleu_score = corpus_bleu(references, predictions, smoothing_function=smoothing)

        # Calculate ROUGE scores
        rouge_scores = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
        for pred, ref in zip(predictions, references):
            pred_text = ' '.join(pred)
            ref_text = ' '.join(ref[0])
            scores = rouge_scorer_obj.score(ref_text, pred_text)
            rouge_scores['rouge1'] += scores['rouge1'].fmeasure
            rouge_scores['rouge2'] += scores['rouge2'].fmeasure
            rouge_scores['rougeL'] += scores['rougeL'].fmeasure

        # Average ROUGE scores
        num_samples = len(predictions)
        for key in rouge_scores:
            rouge_scores[key] /= num_samples

        # Calculate Perplexity with individual processing
        total_loss = 0
        total_length = 0

        for item in test_dataset:
            try:
                # Format input for the model
                input_text = f"<Human>: {item['Question'].strip()}\n<AI>: {item['Answer'].strip()}"
                
                # Tokenize
                inputs = tokenizer(
                    input_text,
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                    max_length=512
                ).to(model.device)
                
                labels = inputs['input_ids'].clone()
                
                # Forward pass
                with torch.no_grad():
                    outputs = model(
                        input_ids=inputs['input_ids'],
                        attention_mask=inputs['attention_mask'],
                        labels=labels
                    )
                
                total_loss += outputs.loss.item()
                total_length += 1
            
            except Exception as e:
                print(f"Error in perplexity calculation for sample: {str(e)}")
                continue

        if total_length == 0:
            perplexity = float('inf')
        else:
            perplexity = torch.exp(torch.tensor(total_loss / total_length))

        # Compile results
        metrics = {
            'bleu': bleu_score,
            'rouge1': rouge_scores['rouge1'],
            'rouge2': rouge_scores['rouge2'],
            'rougeL': rouge_scores['rougeL'],
            'perplexity': perplexity.item() if torch.is_tensor(perplexity) else perplexity,
            'processed_samples': len(predictions),
            'skipped_samples': skipped_samples
        }

        return metrics

    except Exception as e:
        print(f"Error calculating metrics: {str(e)}")
        raise

In [10]:
print("start")

start


In [12]:
try:
    metrics = compute_metrics(model, test_dataset, tokenizer)
    
    print("\nEvaluation Metrics:")
    print(f"BLEU Score: {metrics['bleu']:.4f}")
    print(f"ROUGE-1 F1: {metrics['rouge1']:.4f}")
    print(f"ROUGE-2 F1: {metrics['rouge2']:.4f}")
    print(f"ROUGE-L F1: {metrics['rougeL']:.4f}")
    print(f"Perplexity: {metrics['perplexity']:.4f}")

except Exception as e:
    print(f"An error occurred: {e}")



Processed 448 samples successfully
Skipped 0 samples due to errors

Evaluation Metrics:
BLEU Score: 0.0168
ROUGE-1 F1: 0.2332
ROUGE-2 F1: 0.0530
ROUGE-L F1: 0.1152
Perplexity: 4.5244
